# Task A -- MuRIL embeddings plus SVM

This experiment freezes MuRIL, extracts masked mean+max embeddings from the cleaned
and demojized comments, and trains an RBF SVM on top. It first measures the method
on the repository's fixed 85/15 stratified holdout, then refits the SVM on all 6,401
deduplicated labelled rows and creates a validation submission ZIP.

The holdout is needed only to measure the method. The final SVM fit uses 100% of the
labelled corpus. This is different from the failed TF-IDF ensemble: the SVM here
operates on MuRIL semantic embeddings rather than sparse TF-IDF features.

Expected runtime is approximately 20--45 minutes on a T4, depending on SVM kernel
support-vector count. Upload this notebook to Kaggle, enable GPU and Internet, and
choose Save Version -> Save & Run All.

In [ ]:
import os, pathlib, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

import torch
assert torch.cuda.is_available(), "no GPU -- select a CUDA-enabled runtime"
print("gpu:", torch.cuda.get_device_name(0))


## 1. Extract MuRIL embeddings and evaluate the fixed holdout

In [ ]:
TAG = "task_a_muril_embeddings_svm_rbf"
run_log = pathlib.Path("artifacts/logs") / f"{TAG}.log"
cmd = [sys.executable, "-u", "-m", "hastika.models.muril_embeddings_svm",
       "--tag", TAG, "--kernel", "rbf", "--C", "2.0",
       "--batch-size", "32", "--max-len", "128",
       "--valid-size", "0.15", "--seed", "42"]
print("$", " ".join(cmd), flush=True)
with open(run_log, "w") as fh:
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        fh.write(line)
    p.wait()
if p.returncode:
    raise RuntimeError(f"exit {p.returncode}: {cmd}")
print("embedding-SVM experiment completed")


## 2. Validate and preserve the full-data submission

The runner has already refit the SVM on all labelled rows after scoring the holdout.

In [ ]:
pred = pathlib.Path("artifacts/runs") / TAG / "predictions.csv"
assert pred.exists(), pred
ZIP = pathlib.Path("/kaggle/working/task_a_muril_embeddings_svm_rbf.zip")
subprocess.run([sys.executable, "-m", "hastika.common.submission",
                "--task", "a", "--pred", str(pred),
                "--out", str(ZIP)], check=True)
with zipfile.ZipFile(ZIP) as z:
    assert z.namelist() == ["predictions.csv"], z.namelist()
OUT = pathlib.Path("/kaggle/working/task_a_muril_embeddings_svm_outputs")
OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(ZIP, OUT / ZIP.name)
for name in ["predictions.csv", "test_decision.npy", "test_embeddings.npy",
             "config.json", "holdout_idx.npy", "holdout_decision.npy"]:
    path = pathlib.Path("artifacts/runs") / TAG / name
    if path.exists():
        shutil.copy2(path, OUT / name)
shutil.copy2(run_log, OUT / run_log.name)
print("READY TO UPLOAD:", ZIP)
print("download directory:", OUT)
